imports

In [1]:
import arcgis
from arcgis.gis import GIS
from arcgis.map import Map as webmap
from arcgis.apps.storymap.collection import Collection 
from arcgis.apps.mapshowcase import (
    MapShowcase,
    HighlightedArea,
    HighlightStyle,
    HighlightStyleType,
    HighlightMask,
    HighlightMaskTheme,
    HighlightBlur,
)
import arcgis.geometry as arcGeo
import time
import getpass
from arcgis.gis import GIS, Item
from arcgis.apps.storymap import StoryMap, story, story_content
from arcgis.apps.storymap.story_content import Image, TextStyles, Video, Audio, Embed, Map, Cover, Text, Button, Sidecar, SidecarSlide, Scales
from bs4 import BeautifulSoup, NavigableString
from pathlib import Path
import copy
import shutil
import requests
import json
from pathlib import Path
publish = True
override = False

state abbreviation dictionary

In [2]:
# dictionary to map state names to two letter abbreviations, used for updating the StoryMap collection's thumbnail with the state flag
states_dict = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "Virgin Islands, U.S.": "VI",
}

logging into portal

In [3]:
portal_url = "https://urbanobservatory.maps.arcgis.com/"
portal_username = "wponcy_UO"
portal_password = "YoustayformeUO2025" 
gis = GIS(
    url=portal_url,
    username=portal_username,
    password=portal_password
)
if gis:
    print("Succesful login!")
else:
    print(f"Unable to login to with the credentials provided with the following error: {e}. Please try again.\n")

Succesful login!


In [4]:
def does_url_exist(url: str) -> bool:
    try:
        # Use HEAD request and follow redirects
        response = requests.head(url, allow_redirects=True, timeout=5)
        # Returns True if status code is between 200 and 399
        return response.status_ok 
    except requests.RequestException:
        return False

In [5]:
def build_panel_from_description(title, description):

    panel = [Text(title, style=TextStyles.HEADING)]

    soup = BeautifulSoup(description or "", "html.parser")

    root = soup.body if soup.body else soup

    # print("root", root.contents)
    BLOCK_TAGS = {
        "p",
        "font",
        "h1",
        "h2",
        "h3",
        "h4",
        "ul",
        "ol",
        "img"
    }
    INLINE_TAGS = {
        "a",
        "strong",
        "em",
        "span",
        "br"
    }
    
    def process_node(node, panel):
        if isinstance(node, NavigableString):
            return
    
        tag = node.name.lower()
    
        # Paragraphs
        if tag in ["p", "font"]:

            html = node.decode_contents().strip()
        
            if html:
                panel.append(
                    Text(
                        html,
                        style=TextStyles.PARAGRAPH
                    )
                )

        # Headings
        elif tag in ["h1", "h2", "h3", "h4"]:
            text = node.decode_contents().strip()
    
            if text:
                panel.append(
                    Text(text, style=TextStyles.SUBHEADING)
                )
    
        # Images
        elif tag == "img":
            
            print("Found image:", node)
            print("src:", node.get("src"))
            src = node.get("src")
    
            if src:
                try:
                    panel.append(Image(src))
                except Exception as ex:
                    print(f"Could not add image: {src}")
                    print(ex)
    
        # Unordered lists
        elif tag == "ul":
            bullets = []
    
            for li in node.find_all("li", recursive=False):
                bullets.append(f"• {li.get_text(' ', strip=True)}")
    
            if bullets:
                panel.append(
                    Text("\n".join(bullets), style=TextStyles.PARAGRAPH)
                )
    
        # Ordered lists
        elif tag == "ol":
            bullets = []
    
            for index, li in enumerate(
                node.find_all("li", recursive=False),
                start=1
            ):
                bullets.append(
                    f"{index}. {li.get_text(' ', strip=True)}"
                )
    
            if bullets:
                panel.append(
                    Text("\n".join(bullets), style=TextStyles.PARAGRAPH)
                )

        elif tag == "div": 

            block_children = node.find(BLOCK_TAGS)
        
            if block_children:
                for child in node.children:
                    process_node(child, panel)
            else:
                html = node.decode_contents().strip()
        
                if html:
                    panel.append(
                        Text(html, style=TextStyles.PARAGRAPH)
                    )
            
        elif tag in ["span"]:
            for child in node.children:
                process_node(child, panel)
    
    for child in root.children:
        process_node(child, panel)
    
                
    # printing the panel's information for debug
    print("\tPanel length:", len(panel))

    # for i, item in enumerate(panel):
    #     print(i, type(item))

    # prompting user for end-of-description text
    end_text = input("\tType in any additional text you'd like to add to the sidecar text. "
    "Otherwise, just hit enter for the default: 'Click on the map for details'"
    ) if override else "Click on the map for details."
    panel.append(Text(end_text, style=TextStyles.PARAGRAPH))
    
    return panel

In [6]:
# helper function to rename thumbnails 
def renameThumbnailPath(item_from_showcase):
    cleaned_title = item_from_showcase.showcase_title.replace(" ","_").replace("/", "").replace(".", "").replace("?", "").replace(":", "") # removing spaces
    showcase_item_thumbnail_path = Path(item_from_showcase.download_thumbnail(save_folder=f"/arcgis/home/thumbnails/{cleaned_title}")) # saving thumbnail to a custom folder, retuning a path object
    new_path = str(showcase_item_thumbnail_path.with_name(f"{cleaned_title}.png")) # generating a new name due to API's default thumbnail name, and converting to string
    shutil.move(str(showcase_item_thumbnail_path), new_path) # renaming the thumbnail file in-place
    return new_path

In [7]:
def createStoryMapforShowcaseItem(showcase_item):

    underlying_item = gis.content.get(showcase_item.item_id)
    print(f'\nCreating StoryMap for map: {showcase_item.item.title}')
    new_story = StoryMap()

    title = showcase_item.showcase_title
    description = showcase_item.showcase_description

    # filling in the header & byline
    print('\tFilling in header & byline')
    new_story.contents[0].title = title
    
    byline = input("\tType in any text you'd like for the StoryMap byline. "
    "Otherwise, just hit enter for the default: 'ArcGIS Living Atlas'"
    ) if override else "ArcGIS Living Atlas"

    new_story.contents[0].summary = byline
    new_story.contents[0].date = 'current-date'

    print('\tCreating map for sidecar')
    sidecar_map = Map(showcase_item.item_id) # Creating a map using the underlying portal item

    print('\tUnionizing geometries')
    geometries_union = arcGeo.union([geo.geometry for geo in showcase_item.highlighted_area.graphics])[0] # this merges the (often) multipart geometries for a higlight area   
    
    print('\tSetting map viewpoint')
    sidecar_map.set_viewpoint(
    extent={
        'xmin': geometries_union.extent[0],
        'ymin': geometries_union.extent[1],
        'xmax': geometries_union.extent[2],
        'ymax': geometries_union.extent[3],
        'spatialReference': geometries_union.spatial_reference,
    }, scale=Scales.STATES)
    
    print('\tAssembling sidecar')
    narrative_panel = build_panel_from_description(title, description)
    
    button = Button(
        link= f'{portal_url}/home/item.html?id={showcase_item.item_id}#overview', 
        text="Original Item Page"
    )
    narrative_panel.append(button)
    slide = SidecarSlide(content=narrative_panel,media=sidecar_map)
    sidecar = Sidecar(style="docked-panel")
    sidecar.slides = [slide]
    new_story.add(sidecar)

    print('\tShowing and pinning legend')
    sidecar_map.show_legend = True
    sidecar_map.legend_pinned = True
    
    print('\tSaving storymap')
    storymap_item = new_story.save(title=title, access="public", publish=publish) 
    print('\tStoryMap saved to item ID', storymap_item.id)

    new_path = renameThumbnailPath(showcase_item)
    
    print(f'Thumbnail for showcase item: {title} saved to path: {new_path}')
    return storymap_item, new_path # passing the thumbnail path

In [22]:
def createCollectionFromStoryMaps(list_of_storymaps):

    print(" - " * 30)
    print("\nCreating a StoryMap collection")
    coll = Collection()
    focus_geography = input("Please input a name of for the state of focus:")
    coll.title = f'{focus_geography} Map Portfolio' 
    coll.content[0].title = f'{focus_geography} Map Portfolio' 
    coll.content[0].summary = f'This collection of maps enables {focus_geography} decision-makers to monitor conditions across multiple topics.'
    
    # adding storymaps to collction
    for sm, thumbnail_path in list_of_storymaps:
        print(f"Adding StoryMap '{sm.title}' to collection with thumbnail: {thumbnail_path}.")
        coll.add(item=sm, thumbnail=thumbnail_path) # the add method expects a thumbnail string

    
    cover_path = input("Please input a URL for a Cover image, or just hit enter to skip.")
    if cover_path:
        coll_cover = coll.content[0]
        try:
            cover_img = Image(path = cover_path)
            coll_cover.media = cover_img
        except Exception as e:
            print('Error updating StoryMap Collection cover with error:', e)
    
        
    topic = input("Please input a topic for the StoryMap Collection")
    
    collection_title = f"{focus_geography} {topic} StoryMap Collection" if topic else f"{focus_geography} StoryMap Collection" 
    
    # saving collection
    coll_item = coll.save(title=collection_title, access='public', publish=publish)
    print("Collection saved to item ID:", coll_item.id)

    # updating the collection's thumbnail after saving
    try:
        state_thumbnail_url = f"https://github.com/newtdobbs/State_Flags_png/blob/main/{states_dict[focus_geography].lower()}.png?raw=true"
        does_url_exist(state_thumbnail_url)
        coll_item.update_thumbnail(url=state_thumbnail_url)
    except Exception as e:
        print('Error updating thumbnail with message:', e)
    
    return coll_item

In [23]:
def hide_storymap_covers(collection_item):
    collection_item_data = copy.deepcopy(collection_item.get_data()) # copying the JSON into a new dictionary
    
    collection_item_nodes = collection_item_data['nodes'] # this contains the nodes for the StoryMaps within the collection
    collection_ui_key = next((key for key, val in collection_item_nodes.items() if val['type'] == "collection-ui"), None) # the key pertaining to the collection UI
    storymap_list = collection_item_nodes[collection_ui_key]['data']['items'] # list of the storymaps within a collection
    print(f'There are {len(storymap_list)} StoryMaps within the collection.')
    
    # looping through the storymaps and changing hiding their covers
    for storymap_dictionary in storymap_list:
        storymap_dictionary['hideStoryCover'] = True
        
    if collection_item.update(data=collection_item_data): # assigning the updated JSON to the StoryMap Collection, returns a bool
        print('Successful update')
        return collection_item # returning the newly saved collection

In [24]:
all_storymaps = []

showcase = MapShowcase(input('Please paste in an item ID for a map showcase'), gis=gis)
for item in showcase.showcase_items[0:2]:
    current_item_storymap, current_item_thumbnail_path = createStoryMapforShowcaseItem(item)
    all_storymaps.append([current_item_storymap, current_item_thumbnail_path])    

print('Creating a storymap collection')
my_coll = createCollectionFromStoryMaps(all_storymaps)

print('Hiding Storymap covers')
# hide_storymap_covers(my_coll)
my_coll

Please paste in an item ID for a map showcase 40f9f7815581475fa7d2ec34a7ea5748



Creating StoryMap for map: Owned, rented, or vacant housing units
	Filling in header & byline
	Creating map for sidecar
	Unionizing geometries
	Setting map viewpoint
	Assembling sidecar
	Panel length: 6
	Showing and pinning legend
	Saving storymap
	StoryMap saved to item ID 950a075e898e4048b2014b98c3de2ba2
Thumbnail for showcase item: Owned, rented, or vacant housing units saved to path: \arcgis\home\thumbnails\Owned,_rented,_or_vacant_housing_units\Owned,_rented,_or_vacant_housing_units.png

Creating StoryMap for map: Population living below the federal poverty level
	Filling in header & byline
	Creating map for sidecar
	Unionizing geometries
	Setting map viewpoint
	Assembling sidecar
	Panel length: 5
	Showing and pinning legend
	Saving storymap
	StoryMap saved to item ID 5f2339a2ef0c4937b882f4849e3e4d26
Thumbnail for showcase item: Population living below the federal poverty level saved to path: \arcgis\home\thumbnails\Population_living_below_the_federal_poverty_level\Population_liv

Please input a name of for the state of focus: California


Adding StoryMap 'Owned, rented, or vacant housing units' to collection with thumbnail: \arcgis\home\thumbnails\Owned,_rented,_or_vacant_housing_units\Owned,_rented,_or_vacant_housing_units.png.
Adding StoryMap 'Population living below the federal poverty level' to collection with thumbnail: \arcgis\home\thumbnails\Population_living_below_the_federal_poverty_level\Population_living_below_the_federal_poverty_level.png.


Please input a file path for a Cover image, or just hit enter to skip. https://www.stateside.com/sites/default/files/2020-01/WGA_New%20Logo%202017%20%28002%29.JPG
Please input a topic for the StoryMap Collection Mental Health


...

	Type in any additional text you'd like to add to the sidecar text. Otherwise, just hit enter for the default: 'Click on the map for details' 


	Showing and pinning legend
	Saving storymap
	StoryMap saved to item ID 93cf8e98d33f474f9e27c7996fe4bcca
Thumbnail for showcase item: Population living below the federal poverty level saved to path: \arcgis\home\thumbnails\Population_living_below_the_federal_poverty_level\Population_living_below_the_federal_poverty_level.png
Creating a storymap collection
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 

Creating a StoryMap collection


Please input a name of for the state of focus: Alabama


Adding StoryMap 'Owned, rented, or vacant housing units' to collection with thumbnail: \arcgis\home\thumbnails\Owned,_rented,_or_vacant_housing_units\Owned,_rented,_or_vacant_housing_units.png.
Adding StoryMap 'Population living below the federal poverty level' to collection with thumbnail: \arcgis\home\thumbnails\Population_living_below_the_federal_poverty_level\Population_living_below_the_federal_poverty_level.png.
Collection saved to item ID: 3bb613f5a10e460d80b0c2203c453980
Error updating thumbnail with message: 'Response' object has no attribute 'status_ok'
Hiding Storymap covers


<Item title:"Alabama StoryMap Collection" type:StoryMap owner:wponcy_UO>

"C:\Users\wil13413\OneDrive - Esri\Desktop\MapShowcase\WGA\wga-logo.png"

https://www.stateside.com/sites/default/files/2020-01/WGA_New%20Logo%202017%20%28002%29.JPG